# 00 -- Full End-to-End Pipeline

**"How Biased Is Your Regression Model?" -- Chandra Lekha S**

This notebook reproduces every result and figure in the paper in one continuous run.
For step-by-step walkthroughs use notebooks 01-05.

| Section | Paper | Cells |
|---------|-------|-------|
| Data preparation | S.9.1 | 3-4 |
| Baseline 20-model comparison | S.9.2 | 5-7 |
| OVB demonstration | S.4, S.8.3 | 8-10 |
| Fair regression lambda sweep | S.9.3 | 11-13 |
| Segment-level analysis | S.9.4-9.5 | 14-18 |
| Formal fairness metrics | S.5.2 | 19 |
| Save all outputs | -- | 20 |

**Estimated runtime:** ~8-12 min on CPU.


## Cell 1 -- Install dependencies

In [ ]:
# Run once -- skip if already installed
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm openpyxl scipy


## Cell 2 -- Imports and path setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Make src/ importable. If using Colab, clone the repo first.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader     import load_telco_files, merge_telco_data
from src.preprocessing   import prepare
from src.fair_regression import FairLogisticRegression
from src.metrics         import (
    mean_error_by_segment, segment_disparity,
    d_segment_disparity_squared,
    equalized_odds_difference, demographic_parity_difference,
    segment_metrics_table,
)
from src.visualizations  import (
    plot_accuracy_vs_disparity, plot_fairness_accuracy_tradeoff,
    plot_segment_breakdown, plot_accuracy_by_gender,
    plot_roc_by_contract, savefig,
)

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TAB_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete. Root:', PROJECT_ROOT)


---
## Section 1 -- Data Preparation (Paper S.9.1)

Loads the six IBM Telco Excel files, merges them on Customer ID,
removes identifier/geographic columns, encodes categoricals, scales
numerics, and strips leak columns (Churn Score, CLTV, Customer Status).


## Cell 3 -- Load and merge the six Telco files

In [ ]:
# On Colab: DATA_DIR = '/content/Data'
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

print('Loading files from:', DATA_DIR)
frames = load_telco_files(DATA_DIR)

df = merge_telco_data(frames)
print(f'\nMerged dataframe shape: {df.shape}')
df.head(3)


## Cell 4 -- Prepare features, target, and segments

In [ ]:
X, y, segments_df = prepare(df, drop_leaks=True)

X_np     = X.values.astype(np.float64)
y_np     = y.values.astype(np.float64)
segments = segments_df['Contract'].values

print(f'Feature matrix X : {X.shape}')
print(f'Target y         : {y.shape}  (churn rate {y.mean():.2%})')
print('Segments (Contract):')
for seg, cnt in zip(*np.unique(segments, return_counts=True)):
    print(f'  {seg}: {cnt} ({cnt/len(segments):.1%})')


---
## Section 2 -- Baseline 20-Model Comparison (Paper S.9.2)

Cross-validates 20 classifiers. Computes per-model segment disparity
using Contract type as the protected attribute. Produces **Figure 1**.


## Cell 5 -- Define all 20 models

In [ ]:
from sklearn.linear_model   import LogisticRegression, RidgeClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import (RandomForestClassifier, GradientBoostingClassifier,
                                     AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.naive_bayes     import GaussianNB, BernoulliNB
from sklearn.neural_network  import MLPClassifier
import xgboost as xgb
import lightgbm as lgb

models = {
    'Logistic Regression (L2)': LogisticRegression(max_iter=2000, random_state=42),
    'Logistic Regression (L1)': LogisticRegression(penalty='l1', solver='saga', max_iter=2000, random_state=42),
    'Ridge Classifier'        : RidgeClassifier(random_state=42),
    'Decision Tree'           : DecisionTreeClassifier(random_state=42),
    'Random Forest'           : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees'             : ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'       : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost'                : AdaBoostClassifier(n_estimators=100, random_state=42),
    'XGBoost'                 : xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False),
    'LightGBM'                : lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'SVM (RBF)'               : SVC(kernel='rbf',    probability=True, random_state=42),
    'SVM (Linear)'            : SVC(kernel='linear', probability=True, random_state=42),
    'k-NN (k=5)'              : KNeighborsClassifier(n_neighbors=5),
    'k-NN (k=10)'             : KNeighborsClassifier(n_neighbors=10),
    'Gaussian NB'             : GaussianNB(),
    'Bernoulli NB'            : BernoulliNB(),
    'MLP (2 layers)'          : MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    'MLP (1 layer)'           : MLPClassifier(hidden_layer_sizes=(100,),    max_iter=500, random_state=42),
}
print(f'Defined {len(models)} models.')


## Cell 6 -- Cross-validate global metrics + segment disparity

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics         import make_scorer, precision_score, recall_score, f1_score

def cv_scores(model, X, y, cv=5):
    scoring = {
        'accuracy' : 'accuracy',
        'precision': make_scorer(precision_score, zero_division=0),
        'recall'   : make_scorer(recall_score,    zero_division=0),
        'f1'       : make_scorer(f1_score,        zero_division=0),
    }
    res = cross_validate(
        model, X, y,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        scoring=scoring,
    )
    return {m: float(np.mean(res[f'test_{m}'])) for m in scoring}

results_list  = []
disparity_map = {}

for name, model in models.items():
    print(f'  {name}...')
    try:
        scores = cv_scores(model, X_np, y_np)
        scores['Model'] = name
        results_list.append(scores)
        disparity_map[name] = segment_disparity(model, X_np, y_np, segments)
    except Exception as e:
        print(f'    skipped: {e}')


## Cell 7 -- Assemble results table + Figure 1

In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df[['Model', 'accuracy', 'precision', 'recall', 'f1']]
results_df['disparity'] = results_df['Model'].map(disparity_map)
results_df = (results_df
              .round(4)
              .sort_values('accuracy', ascending=False)
              .reset_index(drop=True))

results_df.to_csv(TAB_DIR / 'baseline_model_comparison.csv', index=False)
print(results_df.to_string(index=False))

# Figure 1
fig1 = plot_accuracy_vs_disparity(results_df)
savefig(fig1, FIG_DIR / 'fig1_accuracy_vs_disparity.png')
plt.show()


---
## Section 3 -- OVB Demonstration (Paper S.4, S.8.3)

Empirically verifies the formula E[beta_hat] = beta + delta*gamma by
comparing full vs reduced logistic regression coefficients.


## Cell 8 -- Select OVB demonstration features

In [ ]:
candidate_features = [
    'Tenure Months',
    'Monthly Charges',
    'Internet Service_Fiber optic',
    'Avg Monthly GB Download',
]
available = [f for f in candidate_features if f in X.columns]
print('Features available for OVB demo:', available)

X_ovb = X[available].copy()
y_ovb = y.copy()


## Cell 9 -- Fit full model and reduced model

In [ ]:
from sklearn.linear_model import LogisticRegression as LR

# Full model
lr_full = LR(max_iter=1000)
lr_full.fit(X_ovb, y_ovb)
coef_full = pd.Series(lr_full.coef_[0], index=available)

# Reduced model -- omit one variable (analogous to omitting NetworkStrain)
candidate_omit = 'Avg Monthly GB Download'
X_red = X_ovb.drop(columns=[candidate_omit])
lr_red = LR(max_iter=1000)
lr_red.fit(X_red, y_ovb)
coef_red = pd.Series(lr_red.coef_[0], index=X_red.columns)

print('Full-model coefficients:')
print(coef_full.round(4).to_string())
print(f'\nReduced-model coefficients ("{candidate_omit}" omitted):')
print(coef_red.round(4).to_string())


## Cell 10 -- Verify the OVB formula  E[beta_hat] = beta + delta*gamma

In [ ]:
from sklearn.linear_model import LinearRegression

# delta: regression of omitted on each included variable
delta_model = LinearRegression()
delta_model.fit(X_red, X_ovb[candidate_omit])
delta = pd.Series(delta_model.coef_, index=X_red.columns)

# gamma: coefficient of the omitted variable in the full model
gamma = coef_full[candidate_omit]

predicted_bias = delta * gamma
actual_bias    = coef_red - coef_full[coef_red.index]

print(f'gamma (full-model coef of "{candidate_omit}"): {gamma:.4f}')
print()
print(pd.DataFrame({
    'delta (omitted -> included)': delta.round(4),
    'Predicted bias  delta*gamma': predicted_bias.round(4),
    'Actual bias  (red - full)'  : actual_bias.round(4),
}))


---
## Section 4 -- Fair Logistic Regression -- lambda Sweep (Paper S.8, S.9.3)

Trains FairLogisticRegression across lambda_f in {0, 0.1, 0.5, 1.0, 2.0, 5.0}
and plots the Pareto frontier. Produces **Figure 2**.


## Cell 11 -- Sweep lambda_fairness

In [ ]:
from sklearn.metrics import accuracy_score

lambda_f_values = [0, 0.1, 0.5, 1.0, 2.0, 5.0]
results_fair = []

for lf in lambda_f_values:
    print(f'  lambda_f = {lf}...')
    model = FairLogisticRegression(
        lambda_complexity=0.1,
        lambda_fairness=lf,
        segments=segments,
    )
    model.fit(X_np, y_np)
    y_proba = model.predict_proba(X_np)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    acc     = accuracy_score(y_np, y_pred)
    errors  = mean_error_by_segment(y_np, y_proba, segments)
    disp    = max(errors.values()) - min(errors.values())
    results_fair.append({'lambda_f': lf, 'accuracy': acc, 'disparity': disp})

fair_df = pd.DataFrame(results_fair)
fair_df.to_csv(TAB_DIR / 'fairness_lambda_sweep.csv', index=False)
print()
print(fair_df.round(4).to_string(index=False))


## Cell 12 -- Figure 2: Fairness-Accuracy Trade-off

In [ ]:
fig2 = plot_fairness_accuracy_tradeoff(fair_df)
savefig(fig2, FIG_DIR / 'fig2_fairness_accuracy_tradeoff.png')
plt.show()


## Cell 13 -- Train the final fair model at lambda_f = 2.0

In [ ]:
fair_model = FairLogisticRegression(
    lambda_complexity=0.1,
    lambda_fairness=2.0,
    segments=segments,
)
fair_model.fit(X_np, y_np)
fair_proba = fair_model.predict_proba(X_np)[:, 1]
fair_class = (fair_proba >= 0.5).astype(int)

print('Fair model (lambda_f=2.0) -- global accuracy:',
      f'{accuracy_score(y_np, fair_class):.4f}')


---
## Section 5 -- Segment-Level Analysis (Paper S.9.4 and S.9.5)

Breaks down model behavior by Contract type and Gender.
Produces **Figures 3, 4, and 5**.


## Cell 14 -- Build the analysis dataframe

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting baseline for comparison alongside the fair model
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_np, y_np)
gb_proba = gb.predict_proba(X_np)[:, 1]
gb_class = (gb_proba >= 0.5).astype(int)

df_analysis = segments_df.copy()
df_analysis['y_true']          = y_np
# Gradient Boosting (baseline)
df_analysis['y_pred_proba']    = gb_proba
df_analysis['y_pred_class']    = gb_class
df_analysis['residual']        = y_np - gb_proba
# Fair Logistic (lambda=2)
df_analysis['fair_pred_proba'] = fair_proba
df_analysis['fair_pred_class'] = fair_class
df_analysis['fair_residual']   = y_np - fair_proba

df_analysis.to_csv(TAB_DIR / 'df_analysis.csv', index=False)
print(f'df_analysis shape: {df_analysis.shape}')
df_analysis.head(3)


## Cell 15 -- Figure 3: Segment-Level Accuracy and Mean Error

In [ ]:
contract_metrics = segment_metrics_table(
    df_analysis, 'Contract',
    y_pred_class_col='y_pred_class',
    residual_col='residual',
)
print('Baseline model -- by Contract type:')
print(contract_metrics.round(4).to_string(index=False))

fig3 = plot_segment_breakdown(contract_metrics)
savefig(fig3, FIG_DIR / 'fig3_segment_breakdown.png')
plt.show()


## Cell 16 -- Figure 4: Accuracy by Gender

Expected finding: near-parity (~0.45 pp gap). Gender alone does not
drive significant disparity in this model.


In [ ]:
gender_metrics = segment_metrics_table(
    df_analysis, 'Gender',
    y_pred_class_col='y_pred_class',
    residual_col='residual',
)
print('By Gender:')
print(gender_metrics.round(4).to_string(index=False))

fig4 = plot_accuracy_by_gender(gender_metrics)
savefig(fig4, FIG_DIR / 'fig4_accuracy_by_gender.png')
plt.show()

gap = abs(gender_metrics['accuracy'].max() - gender_metrics['accuracy'].min())
print(f'\nGender accuracy gap: {gap:.4f} ({gap*100:.2f} percentage points)')


## Cell 17 -- Figure 5: ROC Curves by Contract Type

In [ ]:
fig5 = plot_roc_by_contract(
    df_analysis,
    proba_col='fair_pred_proba',
    title='ROC Curves by Contract - Fair Model (lambda=2)',
)
savefig(fig5, FIG_DIR / 'fig5_roc_by_contract.png')
plt.show()

from sklearn.metrics import roc_auc_score
for contract in sorted(df_analysis['Contract'].unique()):
    mask = df_analysis['Contract'] == contract
    auc_val = roc_auc_score(df_analysis.loc[mask, 'y_true'],
                            df_analysis.loc[mask, 'fair_pred_proba'])
    print(f'  AUC ({contract}): {auc_val:.3f}')


## Cell 18 -- Baseline vs Fair model: segment comparison

In [ ]:
print('=' * 62)
print('Baseline (Gradient Boosting) vs Fair Logistic (lambda=2.0)')
print('=' * 62)

for contract in sorted(df_analysis['Contract'].unique()):
    mask = df_analysis['Contract'] == contract
    sub  = df_analysis[mask]
    acc_base = (sub['y_pred_class']    == sub['y_true']).mean()
    acc_fair = (sub['fair_pred_class'] == sub['y_true']).mean()
    err_base = sub['residual'].mean()
    err_fair = sub['fair_residual'].mean()
    print(f'\n{contract}')
    print(f'  Baseline: acc={acc_base:.4f}  mean_error={err_base:+.4f}')
    print(f'  Fair    : acc={acc_fair:.4f}  mean_error={err_fair:+.4f}')

# Overall disparity
e_base = mean_error_by_segment(df_analysis['y_true'], df_analysis['y_pred_proba'],    df_analysis['Contract'])
e_fair = mean_error_by_segment(df_analysis['y_true'], df_analysis['fair_pred_proba'], df_analysis['Contract'])
d_base = max(e_base.values()) - min(e_base.values())
d_fair = max(e_fair.values()) - min(e_fair.values())

print(f'\nOverall segment disparity:')
print(f'  Baseline  : {d_base:.4f}')
print(f'  Fair (l=2): {d_fair:.4f}  ({(d_base-d_fair)/d_base:.1%} reduction)')


---
## Section 6 -- Formal Fairness Metrics (Paper S.5.2)


## Cell 19 -- DP_diff, EO_diff, D_segment_disparity

In [ ]:
print('Fairness metrics -- Fair Logistic (lambda=2), protected: Contract')

dp   = demographic_parity_difference(df_analysis['fair_pred_class'], df_analysis['Contract'])
eo   = equalized_odds_difference(df_analysis['y_true'], df_analysis['fair_pred_class'], df_analysis['Contract'])
d_sq = d_segment_disparity_squared(df_analysis['y_true'], df_analysis['fair_pred_proba'], df_analysis['Contract'])

print(f'  DP_diff  (Demographic Parity Difference): {dp:.4f}')
print(f'  EO_diff  (Equalized Odds Difference)    : {eo:.4f}')
print(f'  D^2_seg  (Closed-form, Section 8.3)     : {d_sq:.6f}')

print()
print('Fairness metrics -- by Gender:')
dp_g = demographic_parity_difference(df_analysis['fair_pred_class'], df_analysis['Gender'])
eo_g = equalized_odds_difference(df_analysis['y_true'], df_analysis['fair_pred_class'], df_analysis['Gender'])
print(f'  DP_diff: {dp_g:.4f}')
print(f'  EO_diff: {eo_g:.4f}')


---
## Cell 20 -- Save all outputs

In [ ]:
X.to_csv(TAB_DIR / 'X_features.csv',  index=False)
y.to_csv(TAB_DIR / 'y_target.csv',    index=False)
segments_df.to_csv(TAB_DIR / 'segments.csv', index=False)
results_df.to_csv(TAB_DIR / 'baseline_model_comparison.csv', index=False)
fair_df.to_csv(TAB_DIR / 'fairness_lambda_sweep.csv',       index=False)
df_analysis.to_csv(TAB_DIR / 'df_analysis.csv',             index=False)

print('Tables saved:')
for f in sorted(TAB_DIR.glob('*.csv')): print(f'  {f.name}')
print('\nFigures saved:')
for f in sorted(FIG_DIR.glob('*.png')): print(f'  {f.name}')


---
## Summary

This notebook reproduces every empirical claim and figure in the paper:

| Output | Paper reference |
|--------|-----------------|
| fig1_accuracy_vs_disparity.png | Figure 1 -- S.9.2 |
| fig2_fairness_accuracy_tradeoff.png | Figure 2 -- S.9.3 |
| fig3_segment_breakdown.png | Figure 3 -- S.9.4 |
| fig4_accuracy_by_gender.png | Figure 4 -- S.9.4 |
| fig5_roc_by_contract.png | Figure 5 -- S.9.5 |
| baseline_model_comparison.csv | Table behind Figure 1 |
| fairness_lambda_sweep.csv | Table 3 -- S.8.3 |
| df_analysis.csv | All per-customer predictions |

For step-by-step walkthroughs, open notebooks 01-05.
